In [1]:
import glob
import os
import h5py
import numpy as np
from astropy.cosmology import Planck18
import astropy.units as u
import bilby
import json
from bilby.gw.conversion import (
    symmetric_mass_ratio_to_mass_ratio,
    lambda_1_lambda_2_to_lambda_tilde,
    lambda_1_lambda_2_to_delta_lambda_tilde,
    redshift_to_luminosity_distance,
)

In [2]:
BASE_PATH   = '/ligo/shared-scratch/shiksha.pandey/ce_stm_data'
BNS_PATH    = BASE_PATH + '/bns/networks/'
NSBH_PATH   = BASE_PATH + '/nsbh/networks/'
OUTPUT_FILE = BASE_PATH + 'bns_nsbh_fisher_gaussians.h5'

PARAM_IDX = {
    'Mc':          0,
    'eta':         1,
    'dL':          2,
    'theta':       3,
    'phi':         4,
    'iota':        5,
    'psi':         6,
    'tcoal':       7,
    'Phicoal':     8,
    'chi1z':       9,
    'chi2z':       10,
    'LambdaTilde': 11,
    'deltaLambda': 12,
}

In [3]:
def process_network_file(src_file, parent_group):
    with h5py.File(src_file, 'r') as f:
        detected_mask = f['is_detected'][:]
        snr_all       = f['snr'][:]
        cov_all       = f['covariance'][:]
        sky_all       = f['sky_area_90'][:]

        events_all = {}
        for key in f['event_parameters'].keys():
            events_all[key] = f['event_parameters'][key][:]

        detectors     = f.attrs['detectors']
        snr_threshold = float(f.attrs['snr_threshold'])
        n_total       = int(f.attrs['total_events'])
        n_detected    = int(f.attrs['detected_events'])
        param_idx     = json.loads(f.attrs['parameters_indices'])

    det = np.where(detected_mask.astype(bool))[0]
    assert len(det) == n_detected, f'Mask mismatch: {len(det)} vs {n_detected}'

    net_name = os.path.basename(src_file).replace('network_bns_', '').replace('network_nsbh_', '').replace('.h5', '')
    if net_name in parent_group:
        print(f'  [skip] {net_name} already exists')
        return

    net_grp = parent_group.create_group(net_name)
    net_grp.attrs['detectors']            = detectors
    net_grp.attrs['snr_threshold']        = snr_threshold
    net_grp.attrs['total_events']         = n_total
    net_grp.attrs['detected_events']      = n_detected
    net_grp.attrs['detection_efficiency'] = n_detected / n_total
    net_grp.attrs['source_file']          = os.path.basename(src_file)
    net_grp.attrs['dL_units']             = 'Gpc'

    diag      = np.array([np.diag(cov_all[:, :, i]) for i in det])
    sigma_Mc  = np.sqrt(np.abs(diag[:, param_idx['Mc']]))
    sigma_eta = np.sqrt(np.abs(diag[:, param_idx['eta']]))
    sigma_dL  = np.sqrt(np.abs(diag[:, param_idx['dL']]))
    sigma_psi = np.sqrt(np.abs(diag[:, param_idx['psi']]))
    sigma_LT  = np.sqrt(np.abs(diag[:, param_idx['LambdaTilde']]))

    has_delta = 'deltaLambda' in param_idx
    if has_delta:
        sigma_dLT = np.sqrt(np.abs(diag[:, param_idx['deltaLambda']]))

    Mc_det = events_all['Mc'][det]
    eta    = events_all['eta'][det]
    z      = events_all['z'][det]
    dL     = events_all['dL'][det]
    psi    = events_all['psi'][det]
    m1     = events_all['m1_src'][det]
    m2     = events_all['m2_src'][det]
    L1     = events_all['Lambda1'][det]
    L2     = events_all['Lambda2'][det]

    Mc_src       = Mc_det / (1.0 + z)
    sigma_Mc_src = sigma_Mc / (1.0 + z)

    q       = bilby.gw.conversion.symmetric_mass_ratio_to_mass_ratio(eta)
    deps    = 1e-6
    dq_deta = (
        bilby.gw.conversion.symmetric_mass_ratio_to_mass_ratio(eta + deps)
        - bilby.gw.conversion.symmetric_mass_ratio_to_mass_ratio(eta - deps)
    ) / (2.0 * deps)
    sigma_q = np.abs(dq_deta) * sigma_eta

    # for NSBH, L1=0 (BH), use zeros explicitly for the conversion
    L1_conv = L1 if has_delta else np.zeros_like(L1)
    LT_inj  = bilby.gw.conversion.lambda_1_lambda_2_to_lambda_tilde(L1_conv, L2, m1, m2)
    dLT_inj = bilby.gw.conversion.lambda_1_lambda_2_to_delta_lambda_tilde(L1_conv, L2, m1, m2)

    net_grp.create_dataset('original_indices', data=det)
    net_grp.create_dataset('snr',              data=snr_all[det])
    net_grp.create_dataset('sky_area_90',      data=sky_all[det])
    net_grp.create_dataset('m1_src',           data=m1)
    net_grp.create_dataset('m2_src',           data=m2)
    net_grp.create_dataset('Lambda1',          data=L1)
    net_grp.create_dataset('Lambda2',          data=L2)
    net_grp.create_dataset('z',                data=z)

    for name, mu, sig in [
        ('Mc_src',      Mc_src,   sigma_Mc_src),
        ('eta',         eta,      sigma_eta),
        ('q',           q,        sigma_q),
        ('dL',          dL,       sigma_dL),
        ('LambdaTilde', LT_inj,   sigma_LT),
        ('deltaLambda', dLT_inj,  sigma_dLT if has_delta else np.zeros_like(dLT_inj)),
        ('psi',         psi,      sigma_psi),
    ]:
        net_grp.create_dataset(f'mu_{name}',    data=mu)
        net_grp.create_dataset(f'sigma_{name}', data=sig)

    print(f'  wrote {n_detected} detected events -> {net_name}')

In [4]:
# dry run on one file first
test_file = BNS_PATH + 'network_bns_CE40km_1p0MW_Aplus_coat_10.0hz_CE20km_1p0MW_Aplus_coat_10.0hz_LIA+_10.0hz.h5'
eos_data = np.load('/ligo/home/ligo.org/sanika.khadkikar/Projects/stm/sfho_eos_data.npz', allow_pickle=True)

with h5py.File('test_restructure.h5', 'w') as out:
    eos_grp = out.create_group('eos_metadata')
    eos_grp.attrs['eos_name']   = str(eos_data['eos_name'])
    eos_grp.attrs['M_max_TOV']  = float(eos_data['M_max_TOV'])
    eos_grp.attrs['Lambda_1p4'] = float(eos_data['Lambda_1p4'])
    eos_grp.attrs['Lambda_max'] = float(eos_data['Lambda_max'])
    eos_grp.attrs['R_1p4_km']   = float(eos_data['R_1p4_km'])
    eos_grp.create_dataset('mass_msol', data=eos_data['mass_msol'])
    eos_grp.create_dataset('lambda',    data=eos_data['lambda'])
    eos_grp.create_dataset('radius_km', data=eos_data['radius_km'])
    eos_grp.create_dataset('pressure_gcc',  data=eos_data['pressure_gcc'])
    eos_grp.create_dataset('edensity_gcc',  data=eos_data['edensity_gcc'])

    test_grp = out.create_group('BNS')
    process_network_file(test_file, test_grp)

with h5py.File('test_restructure.h5', 'r') as f:
    print('Top level keys:', list(f.keys()))
    print('EOS attrs:', dict(f['eos_metadata'].attrs))
    print()
    net = list(f['BNS'].keys())[0]
    grp = f['BNS'][net]
    print(f'Network:        {net}')
    print(f'n_detected:     {grp.attrs["detected_events"]}')
    print(f'original_indices[:3]: {grp["original_indices"][:3]}')
    print(f'snr[:3]:        {grp["snr"][:3]}')
    print(f'sky_area_90[:3]:{grp["sky_area_90"][:3]}')
    print(f'm1_src[:3]:     {grp["m1_src"][:3]}')
    print(f'm2_src[:3]:     {grp["m2_src"][:3]}')
    print(f'Lambda1[:3]:    {grp["Lambda1"][:3]}')
    print(f'Lambda2[:3]:    {grp["Lambda2"][:3]}')
    print(f'z[:3]:          {grp["z"][:3]}')
    print()
    for p in ['Mc_src','eta','q','dL','LambdaTilde','deltaLambda','psi']:
        print(f'{p:20s}: mu={grp[f"mu_{p}"][0]:.6f}  sigma={grp[f"sigma_{p}"][0]:.6f}')

  wrote 29253 detected events -> CE40km_1p0MW_Aplus_coat_10.0hz_CE20km_1p0MW_Aplus_coat_10.0hz_LIA+_10.0hz
Top level keys: ['BNS', 'eos_metadata']
EOS attrs: {'Lambda_1p4': np.float64(331.9406061712275), 'Lambda_max': np.float64(2435.164319335637), 'M_max_TOV': np.float64(2.060332319393987), 'R_1p4_km': np.float64(11.90569803500183), 'eos_name': 'SFHo'}

Network:        CE40km_1p0MW_Aplus_coat_10.0hz_CE20km_1p0MW_Aplus_coat_10.0hz_LIA+_10.0hz
n_detected:     29253
original_indices[:3]: [2 3 8]
snr[:3]:        [17.06557335 16.53743642 14.08397921]
sky_area_90[:3]:[121.35823032 436.73274154  38.82754064]
m1_src[:3]:     [1.65422805 1.61927268 2.02453672]
m2_src[:3]:     [1.44499565 1.27726019 1.92824876]
Lambda1[:3]:    [101.75930037 120.73797231  11.44997156]
Lambda2[:3]:    [273.98677338 599.12184115  24.10232187]
z[:3]:          [1.15439516 1.39954065 1.12342509]

Mc_src              : mu=1.345323  sigma=0.000177
eta                 : mu=0.248861  sigma=0.009737
q                   : 

/ligo/home/ligo.org/sanika.khadkikar/.conda/envs/ns_pe/lib/python3.11/site-packages/bilby/gw/conversion.py:914: RuntimeWarning: invalid value encountered in sqrt
  return temp - (temp ** 2 - 1) ** 0.5


In [5]:
with h5py.File('test_restructure.h5', 'r') as f:
    print('EOS attrs:', dict(f['eos_metadata'].attrs))
    print('EOS datasets:', list(f['eos_metadata'].keys()))
    # print('mass_msol:', f['eos_metadata']['mass_msol'][:5])
    # print('lambda:',    f['eos_metadata']['lambda'][:5])
    # print('radius_km:', f['eos_metadata']['radius_km'][:5])

EOS attrs: {'Lambda_1p4': np.float64(331.9406061712275), 'Lambda_max': np.float64(2435.164319335637), 'M_max_TOV': np.float64(2.060332319393987), 'R_1p4_km': np.float64(11.90569803500183), 'eos_name': 'SFHo'}
EOS datasets: ['edensity_gcc', 'lambda', 'mass_msol', 'pressure_gcc', 'radius_km']


In [6]:
BNS_PATH

'/ligo/shared-scratch/shiksha.pandey/ce_stm_data/bns/networks/'

In [7]:
# run everything once dry run looks good
eos_data = np.load('/ligo/home/ligo.org/sanika.khadkikar/Projects/stm/sfho_eos_data.npz', allow_pickle=True)
OUTPUT_FILE = '/ligo/home/ligo.org/sanika.khadkikar/Projects/stm/post_processing_ns/restructured_networks.h5'
bns_files  = sorted(glob.glob(BNS_PATH  + '*.h5'))
nsbh_files = sorted(glob.glob(NSBH_PATH + '*.h5'))

print(f'BNS files:  {len(bns_files)}')
print(f'NSBH files: {len(nsbh_files)}')

with h5py.File(OUTPUT_FILE, 'w') as out:
    eos_grp = out.create_group('eos_metadata')
    eos_grp.attrs['eos_name']   = str(eos_data['eos_name'])
    eos_grp.attrs['M_max_TOV']  = float(eos_data['M_max_TOV'])
    eos_grp.attrs['Lambda_1p4'] = float(eos_data['Lambda_1p4'])
    eos_grp.attrs['Lambda_max'] = float(eos_data['Lambda_max'])
    eos_grp.attrs['R_1p4_km']   = float(eos_data['R_1p4_km'])
    eos_grp.create_dataset('mass_msol', data=eos_data['mass_msol'])
    eos_grp.create_dataset('lambda',    data=eos_data['lambda'])
    eos_grp.create_dataset('radius_km', data=eos_data['radius_km'])
    eos_grp.create_dataset('pressure_gcc', data=eos_data['pressure_gcc'])
    eos_grp.create_dataset('edensity_gcc', data=eos_data['edensity_gcc'])

    bns_grp  = out.create_group('BNS')
    nsbh_grp = out.create_group('NSBH')

    for fpath in bns_files:
        print(os.path.basename(fpath))
        try:
            process_network_file(fpath, bns_grp)
        except Exception as e:
            print(f'  [ERROR] {e}')

    for fpath in nsbh_files:
        print(os.path.basename(fpath))
        try:
            process_network_file(fpath, nsbh_grp)
        except Exception as e:
            print(f'  [ERROR] {e}')

print(f'Done. Output: {OUTPUT_FILE}')

BNS files:  35
NSBH files: 35
network_bns_CE40km_1p0MW_Aplus_coat_10.0hz_CE20km_1p0MW_Aplus_coat_10.0hz_LIA+_10.0hz.h5
  wrote 29253 detected events -> CE40km_1p0MW_Aplus_coat_10.0hz_CE20km_1p0MW_Aplus_coat_10.0hz_LIA+_10.0hz
network_bns_CE40km_1p0MW_Aplus_coat_10.0hz_ETD_5.0hz_LIA+_10.0hz.h5
  wrote 33762 detected events -> CE40km_1p0MW_Aplus_coat_10.0hz_ETD_5.0hz_LIA+_10.0hz
network_bns_CE40km_1p0MW_Aplus_coat_15.0hz_CE20km_1p0MW_Aplus_coat_15.0hz_LIA+_10.0hz.h5
  wrote 23804 detected events -> CE40km_1p0MW_Aplus_coat_15.0hz_CE20km_1p0MW_Aplus_coat_15.0hz_LIA+_10.0hz
network_bns_CE40km_1p0MW_Aplus_coat_15.0hz_ETD_5.0hz_LIA+_10.0hz.h5
  wrote 29483 detected events -> CE40km_1p0MW_Aplus_coat_15.0hz_ETD_5.0hz_LIA+_10.0hz
network_bns_CE40km_1p0MW_Aplus_coat_7.0hz_CE20km_1p0MW_Aplus_coat_7.0hz_LIA+_10.0hz.h5
  wrote 30719 detected events -> CE40km_1p0MW_Aplus_coat_7.0hz_CE20km_1p0MW_Aplus_coat_7.0hz_LIA+_10.0hz
network_bns_CE40km_1p0MW_Aplus_coat_7.0hz_ETD_5.0hz_LIA+_10.0hz.h5
  wrote 348

In [8]:
with h5py.File(OUTPUT_FILE, 'r') as f:
    min_z = float('inf')
    min_source = None
    
    for system_type in ['BNS', 'NSBH']:
        for net_name in f[system_type].keys():
            z_vals = f[system_type][net_name]['z'][:]
            net_min = np.min(z_vals)
            if net_min < min_z:
                min_z = net_min
                min_source = f'{system_type}/{net_name}'
    
    print(f'Lowest z: {min_z:.6f} (from {min_source})')

Lowest z: 0.018068 (from NSBH/CE40km_1p0MW_Aplus_coat_10.0hz_CE20km_1p0MW_Aplus_coat_10.0hz_LIA+_10.0hz)


In [9]:
OUTPUT_FILE = '/ligo/home/ligo.org/sanika.khadkikar/Projects/stm/post_processing_ns/restructured_networks.h5'

with h5py.File(OUTPUT_FILE, 'r') as f:
    all_z = []
    for system_type in ['BNS']:#, 'NSBH']:
        for net_name in f[system_type].keys():
            all_z.extend(f[system_type][net_name]['z'][:])
    
    print(f'Lowest z: {np.min(all_z):.6f}')

Lowest z: 0.042576


In [10]:
from bilby.gw.conversion import redshift_to_luminosity_distance
import astropy.units as u

z = 0.042576
dL = redshift_to_luminosity_distance(z)  # returns value in Mpc by default
print(f'z = {z} → dL = {dL:.2f} Mpc')

z = 0.042576 → dL = 194.49 Mpc


In [11]:
with h5py.File(OUTPUT_FILE, 'r') as f:
    print("BNS networks:", list(f['BNS'].keys()))
    print("NSBH networks:", list(f['NSBH'].keys()))
    print()
    
    # Check a sample network
    bns_sample = list(f['BNS'].keys())[0]
    grp = f['BNS'][bns_sample]
    
    print(f"Sample: {bns_sample}")
    print(f"  detected_events attr: {grp.attrs['detected_events']}")
    print(f"  z dataset length: {len(grp['z'])}")
    print(f"  Do they match? {grp.attrs['detected_events'] == len(grp['z'])}")
    print(f"  z min/max: {np.min(grp['z']):.6f} / {np.max(grp['z']):.6f}")

BNS networks: ['CE40km_1p0MW_Aplus_coat_10.0hz_CE20km_1p0MW_Aplus_coat_10.0hz_LIA+_10.0hz', 'CE40km_1p0MW_Aplus_coat_10.0hz_ETD_5.0hz_LIA+_10.0hz', 'CE40km_1p0MW_Aplus_coat_15.0hz_CE20km_1p0MW_Aplus_coat_15.0hz_LIA+_10.0hz', 'CE40km_1p0MW_Aplus_coat_15.0hz_ETD_5.0hz_LIA+_10.0hz', 'CE40km_1p0MW_Aplus_coat_7.0hz_CE20km_1p0MW_Aplus_coat_7.0hz_LIA+_10.0hz', 'CE40km_1p0MW_Aplus_coat_7.0hz_ETD_5.0hz_LIA+_10.0hz', 'CE40km_1p0MW_aLIGO_coat_10.0hz_CE20km_1p0MW_aLIGO_coat_10.0hz_LIA+_10.0hz', 'CE40km_1p0MW_aLIGO_coat_10.0hz_ETD_5.0hz_LIA+_10.0hz', 'CE40km_1p0MW_aLIGO_coat_15.0hz_CE20km_1p0MW_aLIGO_coat_15.0hz_LIA+_10.0hz', 'CE40km_1p0MW_aLIGO_coat_15.0hz_ETD_5.0hz_LIA+_10.0hz', 'CE40km_1p0MW_aLIGO_coat_7.0hz_CE20km_1p0MW_aLIGO_coat_7.0hz_LIA+_10.0hz', 'CE40km_1p0MW_aLIGO_coat_7.0hz_ETD_5.0hz_LIA+_10.0hz', 'CE40km_1p5MW_Aplus_coat_10.0hz_CE20km_1p5MW_Aplus_coat_10.0hz_ETD_5.0hz', 'CE40km_1p5MW_Aplus_coat_10.0hz_CE20km_1p5MW_Aplus_coat_10.0hz_LIA+_10.0hz', 'CE40km_1p5MW_Aplus_coat_10.0hz_ET2L_5.0h